In [ ]:
# =========================================================
# 1. Import Libraries
# =========================================================
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from googletrans import Translator


# =========================================================
# 2. Load Dataset
# =========================================================
fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat([fake_df, true_df])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


# =========================================================
# 3. Combine Columns
# =========================================================
df["content"] = df["title"] + " " + df["text"]


# =========================================================
# 4. Clean Text 
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]

    return " ".join(words)

df["clean"] = df["content"].apply(clean_text)


# =========================================================
# 5. Train-Test Split
# =========================================================
X = df["clean"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# =========================================================
# 6. TF-IDF Vectorization 
# =========================================================
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2)   
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# =========================================================
# 7. Train Model
# =========================================================
model = LogisticRegression(max_iter=2000)
model.fit(X_train_vec, y_train)


# =========================================================
# 8. Evaluation
# =========================================================
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


# =========================================================
# 9. Translator Setup (FIXED)
# =========================================================
translator = Translator()

def translate_to_english(text):
    try:
        translated = translator.translate(text, dest='en').text
        return translated
    except:
        return text   # fallback


# =========================================================
# 10. Final Prediction Function (FIXED)
# =========================================================
def predict_news(text):

    # Step 1: Translate ALWAYS
    text = translate_to_english(text)

    # Debug (optional)
    print("Translated:", text)

    # Step 2: Clean
    text = clean_text(text)

    # Step 3: Vectorize
    vec = vectorizer.transform([text])

    # Step 4: Predict
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]

    confidence = max(prob)

    # Step 5: Confidence control
    if confidence < 0.6:
        return "UNCERTAIN", confidence

    return ("REAL" if pred == 1 else "FAKE", confidence)


# =========================================================
# 11. Interactive Loop
# =========================================================
while True:
    text = input("Enter News (type 'exit' to stop): ")

    if text.lower() == "exit":
        break

    result, confidence = predict_news(text)

    print("Result:", result)
    print("Confidence:", round(confidence, 3))
    print("-" * 40)


# =========================================================
# 12. Save Model
# =========================================================
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

Accuracy: 0.9879732739420936
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4710
           1       0.98      0.99      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

